In [12]:
import os
import re
import sys
import json

import sys 
sys.path.insert(0, "../../")

from src import PersonalAI, PersonalAIConfig, QAPipelineConfig, MemPipelineConfig, \
            GraphModelConfig, EmbeddingsModelConfig, EmbedderModelConfig

import src.question_answer_rephrasing.QuestionProcessor as qp_qp
import src.question_answer_rephrasing.AnswerProcessor as qap_ap

from src.agents.utils import AbstractAgentConnector, AgentConnectorConfig
from src.agents.connectors import OpenAIConnector, GigaChatConnector

from dotenv import load_dotenv

load_dotenv(".secrets")

True

In [13]:
with open("/Users/matthewiskornev/Documents/skoltech_work/personal_ai_v2/notebooks/data/memory_benchmark_sorted.json", 'r') as f:
    sorted_data = json.load(f)

In [14]:
test_data = []
for i, tmp_dict in enumerate(sorted_data[12:]):
    if i >= 3:
        break

    new_dict = {}
    new_dict['qa'] = tmp_dict['qa'][:3]

    new_dict_conv = {}
    new_dict_conv['speaker_a'] = tmp_dict['conversation']['speaker_a']
    new_dict_conv['speaker_b'] = tmp_dict['conversation']['speaker_b']
    new_dict_conv['session_1'] = tmp_dict['conversation']['session_1'][:5]
    # new_dict_conv['session_2'] = tmp_dict['conversation']['session_2'][:15]

    new_dict['conversation'] = new_dict_conv

    test_data.append(new_dict)

In [15]:
# with open("/Users/matthewiskornev/Documents/skoltech_work/personal_ai_v2/notebooks/data/golden_set_for_rephrase_answers.json", 'w') as f:
#     json.dump(test_data, f, indent=4, ensure_ascii=False)

In [16]:
# with open("/Users/matthewiskornev/Documents/skoltech_work/personal_ai_v2/notebooks/data/golden_set_for_rephrase_answers.json", 'r') as f:
#     test_data = json.load(f)

In [17]:
test_data

[{'qa': [{'question': 'Как зовут мою подругу, которая увлекается стендапом?',
    'answer': 'Вашу подругу зовут Лида.',
    'evidence': ['D2'],
    'category': 'fact_equal_session'},
   {'question': 'Занимаюсь ли я волейболом?',
    'answer': 'Да.',
    'evidence': ['D7'],
    'category': 'fact_equal_session'},
   {'question': 'Как называется мой мотоцикл?',
    'answer': 'У меня нет такой информации.',
    'evidence': ['D11'],
    'category': 'no_info'}],
  'conversation': {'speaker_a': 'assistant',
   'speaker_b': 'user',
   'session_1': [{'speaker': 'user',
     'dia_id': 'D1:1',
     'text': 'Привет, мне нужны примеры абстракционизма в искусстве.'},
    {'speaker': 'assistant',
     'dia_id': 'D1:2',
     'text': 'Привет! Абстракционизм — направление искусства, которое отвергает точное изображение реальных предметов и направлено на передачу чувств, мыслей и переживаний через формы, линии, цвета и текстуры. Вот несколько известных работ, относящихся к абстрактному искусству:\n\n### 

In [18]:
question_batches = []
batch_size = 7

for i in range(0, len(sorted_data), batch_size):
    question_batches.append(sorted_data[i: i+batch_size])

In [19]:
agent = OpenAIConnector()
# agent = GigaChatConnector()

In [20]:
BATCH_NUM = 4


quest_batch = question_batches[BATCH_NUM]
# quest_batch = test_data

In [21]:
quest_batch[0]

{'qa': [{'question': 'Какие организационные изменения объявили у меня на работе?',
   'answer': 'Сокращения.',
   'evidence': ['D2'],
   'category': 'fact_equal_session'},
  {'question': 'Какого по счёту моего мужа звали Семён?',
   'answer': 'Первого.',
   'evidence': ['D4'],
   'category': 'fact_equal_session'},
  {'question': 'В каких местах я была в Китае?',
   'answer': 'На Хайнане и в Пекине.',
   'evidence': ['D23'],
   'category': 'info_consolidation'},
  {'question': 'За изучение скольких профессий брался мой сын?',
   'answer': 'Трёх.',
   'evidence': ['D20'],
   'category': 'info_consolidation'},
  {'question': 'Какой у меня сейчас семейный статус?',
   'answer': 'В разводе.',
   'evidence': ['D14'],
   'category': 'info_updating'},
  {'question': 'На чём я катаюсь каждый день?',
   'answer': 'На электросамокате.',
   'evidence': ['D18'],
   'category': 'info_updating'},
  {'question': 'Сколько у меня было мужей и мужчин, с которыми были серьезые отношения?',
   'answer': 'Т

In [11]:
Answproc = qap_ap.AnswerProcessor(test_data, agent)
Answproc.run_answer_rephrasing()

[{'qa': [{'question': 'Как зовут мою подругу, которая увлекается стендапом?',
    'answer': 'Вашу подругу зовут Лида.',
    'evidence': ['D2'],
    'category': 'fact_equal_session',
    'rephrased_question': 'Как зовут подругу опльзователя, которая увлекается стендапом?',
    'was_reject_while_rephrasing': False,
    'raw_answer': 'Подругу пользователя зовут Лида.',
    'final_answer': 'Вашу подругу зовут Лида.'},
   {'question': 'Занимаюсь ли я волейболом?',
    'answer': 'Да.',
    'evidence': ['D7'],
    'category': 'fact_equal_session',
    'rephrased_question': 'Занимается ли пользователь волейболом?',
    'was_reject_while_rephrasing': False,
    'raw_answer': 'Да.',
    'final_answer': 'Да.'},
   {'question': 'Как называется мой мотоцикл?',
    'answer': 'У меня нет такой информации.',
    'evidence': ['D11'],
    'category': 'no_info',
    'rephrased_question': 'Как называется мотоцикл пользователя?',
    'was_reject_while_rephrasing': False,
    'raw_answer': 'У меня нет такой

In [22]:
Questproc = qp_qp.QuestProcessor(test_data, agent)
Questproc.run_question_rephrasing()

[{'qa': [{'question': 'Как зовут мою подругу, которая увлекается стендапом?',
    'answer': 'Вашу подругу зовут Лида.',
    'evidence': ['D2'],
    'category': 'fact_equal_session',
    'rephrased_question': 'Как зовут подругу пользователя, которая увлекается стендапом?',
    'was_reject_while_rephrasing': False,
    'raw_answer': None,
    'final_answer': None},
   {'question': 'Занимаюсь ли я волейболом?',
    'answer': 'Да.',
    'evidence': ['D7'],
    'category': 'fact_equal_session',
    'rephrased_question': 'Занимается ли пользователь волейболом?',
    'was_reject_while_rephrasing': False,
    'raw_answer': None,
    'final_answer': None},
   {'question': 'Как называется мой мотоцикл?',
    'answer': 'У меня нет такой информации.',
    'evidence': ['D11'],
    'category': 'no_info',
    'rephrased_question': 'Как называется мотоцикл пользователя?',
    'was_reject_while_rephrasing': False,
    'raw_answer': None,
    'final_answer': None}],
  'conversation': {'speaker_a': 'assi

In [30]:
Questproc = qp_qp.QuestProcessor(quest_batch, agent)
Questproc.run_question_rephrasing()
with open(f"/Users/matthewiskornev/Documents/skoltech_work/personal_ai_v2/notebooks/data/output/questions/quest_batch_{BATCH_NUM}.json", 'w', encoding='utf-8') as f:
    json.dump(quest_batch, f, ensure_ascii=False, indent=4)
Questproc = qp_qp.QuestProcessor(quest_batch, agent)

In [1]:
quest_batch[0]

NameError: name 'quest_batch' is not defined

In [ ]:
# Questproc = qp_qp.QuestProcessor(test_data, agent)